# Regime-Stratified Kronos Fine-Tuning

This notebook fine-tunes the Kronos foundation model on a **per-regime basis** using the RLM DuckDB lake.

**Why this matters:** A single Kronos checkpoint trained on all market conditions averages over regime-specific dynamics. By training a separate checkpoint per HMM regime state we get specialised models that can better capture the return distribution of each regime (calm-trend, volatile-expansion, mean-reverting, etc.).

**Flow:**
1. Load bars (CSV or MicrostructureDB)
2. Run `FactorPipeline` → `HybridForecastPipeline` to annotate HMM regime states
3. Split the bar series by `hmm_state`
4. Fine-tune one Kronos checkpoint per stratum
5. Evaluate val-loss per stratum and export a `regime_metadata.json` for downstream use

**Prerequisites:** `pip install -e ".[kronos,microstructure,ibkr,datalake]"`

In [ ]:
from __future__ import annotations
import json
import logging
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Make repo root importable
ROOT = Path("__file__").resolve().parents[1]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

# Import RLMKlineDataset and the per-epoch train/val helpers from finetune_kronos.py
from finetune_kronos import RLMKlineDataset, _train_one_epoch

from rlm.factors.pipeline import FactorPipeline
from rlm.forecasting.hmm import HMMConfig
from rlm.forecasting.pipeline import HybridForecastPipeline
from rlm.kronos.config import KronosConfig
from rlm.kronos.model.kronos import Kronos, KronosTokenizer
from rlm.types.forecast import ForecastConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

## Configuration

Edit the variables below to match your setup.

In [ ]:
# ── User config ────────────────────────────────────────────────────────────
SYMBOL           = "SPY"
HMM_STATES       = 6        # Must match the regime model you plan to deploy
EPOCHS           = 15       # Per-stratum epochs (start low; increase if val loss keeps falling)
BATCH_SIZE       = 8
LEARNING_RATE    = 5e-5
LOOKBACK         = 90       # Context window fed to Kronos
PRED_LEN         = 10       # Horizon the model predicts
MIN_STRATUM_SIZE = 100      # Skip a regime stratum with fewer bars than this
SEED             = 42
OUT_ROOT         = ROOT / "data" / "models" / "kronos" / SYMBOL

# Source for bars — switch to MicrostructureDB if you have the lake built
USE_DUCK_DB = False   # Set True to pull from data/microstructure DuckDB lake
START_DATE  = "2022-01-01"
END_DATE    = "2026-01-01"

# ── Reproducibility ────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cfg_k = KronosConfig.from_yaml()
DEVICE = cfg_k.device
print(f"Symbol: {SYMBOL}  |  HMM states: {HMM_STATES}  |  Device: {DEVICE}")

## 1. Load Bars

In [ ]:
if USE_DUCK_DB:
    from rlm.microstructure.database.query import MicrostructureDB
    db = MicrostructureDB(data_path=str(ROOT / "data" / "microstructure"))
    raw_df = db.load_underlying_bars(SYMBOL, START_DATE, END_DATE)
    db.close()
    raw_df = raw_df.rename(columns={"ts": "timestamp"}) if "ts" in raw_df.columns else raw_df
    print(f"Loaded {len(raw_df):,} bars from DuckDB lake")
else:
    bars_path = ROOT / f"data/raw/bars_{SYMBOL}.csv"
    if not bars_path.is_file():
        raise FileNotFoundError(
            f"Bars not found at {bars_path}.\n"
            "Run: python scripts/build_rolling_backtest_dataset.py "
            f"--fetch-ibkr --symbol {SYMBOL} --start {START_DATE}"
        )
    raw_df = pd.read_csv(bars_path, parse_dates=["timestamp"])
    print(f"Loaded {len(raw_df):,} bars from {bars_path}")

raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)
raw_df.head(3)

## 2. Factor Pipeline + HMM Regime Annotation

In [ ]:
from rlm.datasets.bars_enrichment import prepare_bars_for_factors

# Enrich bars (attach VIX, option-chain aggs if present)
chain_path = ROOT / f"data/raw/option_chain_{SYMBOL}.csv"
opch = (
    pd.read_csv(chain_path, parse_dates=["timestamp", "expiry"])
    if chain_path.is_file()
    else None
)
df_enriched = prepare_bars_for_factors(
    raw_df.set_index("timestamp"),
    opch,
    underlying=SYMBOL,
    attach_vix=True,
)

# Factor pipeline
print("Running FactorPipeline ...")
factor_df = FactorPipeline().run(df_enriched)
print(f"Factors shape: {factor_df.shape}")

In [ ]:
# Fit HMM and annotate regime states
print(f"Fitting HMM({HMM_STATES}) and annotating regimes ...")
fc = ForecastConfig(drift_gamma_alpha=0.65, sigma_floor=1e-4, direction_neutral_threshold=0.3)
forecast_df = HybridForecastPipeline(
    config=fc,
    move_window=100,
    vol_window=100,
    hmm_config=HMMConfig(n_states=HMM_STATES),
).run(factor_df)

print("Regime state distribution:")
print(forecast_df["hmm_state"].value_counts().sort_index())

## 3. Visualise Regime Distribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Close price coloured by regime state
colours = plt.cm.tab10(np.linspace(0, 1, HMM_STATES))
for state in range(HMM_STATES):
    mask = forecast_df["hmm_state"] == state
    axes[0].scatter(
        forecast_df.index[mask],
        forecast_df.loc[mask, "close"],
        s=3, color=colours[state], label=f"state {state}",
    )
axes[0].set_ylabel("Close")
axes[0].set_title(f"{SYMBOL} — Close price coloured by HMM state")
axes[0].legend(loc="upper left", markerscale=4, fontsize=8)

# HMM state over time
axes[1].plot(forecast_df.index, forecast_df["hmm_state"], lw=0.8, color="steelblue")
axes[1].set_ylabel("HMM state")
axes[1].set_title("HMM state over time")

plt.tight_layout()
plt.show()

## 4. Split Bars by Regime State

We align the original bars back to the annotated states (HMM is fit on factor_df but we fine-tune Kronos on raw OHLCV).

In [ ]:
# Merge hmm_state back into the original raw bar index
bars_with_state = raw_df.copy()
bars_with_state = bars_with_state.set_index("timestamp") if "timestamp" in bars_with_state.columns else bars_with_state
bars_with_state["hmm_state"] = forecast_df["hmm_state"].reindex(bars_with_state.index)
bars_with_state = bars_with_state.dropna(subset=["hmm_state"]).copy()
bars_with_state["hmm_state"] = bars_with_state["hmm_state"].astype(int)

strata: dict[int, pd.DataFrame] = {}
for state in range(HMM_STATES):
    sub = bars_with_state[bars_with_state["hmm_state"] == state].copy()
    if len(sub) >= MIN_STRATUM_SIZE:
        strata[state] = sub.reset_index()
        label = (
            forecast_df.loc[forecast_df["hmm_state"] == state, "hmm_state_label"]
            .mode()
            .iloc[0]
            if "hmm_state_label" in forecast_df.columns
            else f"state_{state}"
        )
        print(f"  state {state:2d}  {label:35s}  {len(sub):5,} bars  -> INCLUDED")
    else:
        print(f"  state {state:2d}  (only {len(sub)} bars)  -> SKIPPED (< {MIN_STRATUM_SIZE})")

print(f"\nStrata to fine-tune: {sorted(strata.keys())}")

## 5. Fine-Tuning Loop per Stratum

In [ ]:
def _validate_epoch(
    tokenizer: KronosTokenizer,
    model: Kronos,
    loader: DataLoader,
    device: str,
) -> float:
    """Compute validation loss without gradient updates."""
    tokenizer.eval()
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for x_batch, stamp_batch in loader:
            x_batch = x_batch.to(device)
            stamp_batch = stamp_batch.to(device)
            (_, recon), bsq_loss, _, z_indices = tokenizer(x_batch)
            recon_loss = nn.functional.mse_loss(recon, x_batch)
            s1_ids, s2_ids = z_indices[0], z_indices[1]
            s1_logits, s2_logits = model(
                s1_ids[:, :-1], s2_ids[:, :-1], stamp_batch[:, :-1],
                use_teacher_forcing=True, s1_targets=s1_ids[:, 1:],
            )
            pred_loss, _, _ = model.head.compute_loss(
                s1_logits, s2_logits, s1_ids[:, 1:], s2_ids[:, 1:]
            )
            total += (recon_loss + bsq_loss + pred_loss).item()
            n += 1
    return total / max(n, 1)


def finetune_stratum(
    state: int,
    df: pd.DataFrame,
    *,
    cfg_k: KronosConfig,
    out_root: Path,
    epochs: int = EPOCHS,
    batch_size: int = BATCH_SIZE,
    lr: float = LEARNING_RATE,
    lookback: int = LOOKBACK,
    pred_len: int = PRED_LEN,
    device: str = DEVICE,
) -> dict:
    """Fine-tune Kronos on a single regime stratum and return metadata."""
    split = int(len(df) * 0.85)
    train_ds = RLMKlineDataset(df.iloc[:split], lookback=lookback, pred_len=pred_len)
    val_ds   = RLMKlineDataset(df.iloc[split:],  lookback=lookback, pred_len=pred_len)

    if len(train_ds) == 0:
        logger.warning("Stratum %d: no training samples after windowing — skipping.", state)
        return {}

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)

    tokenizer = KronosTokenizer.from_pretrained(cfg_k.tokenizer_name).to(device)
    model     = Kronos.from_pretrained(cfg_k.model_name).to(device)

    optimizer = torch.optim.AdamW(
        list(tokenizer.parameters()) + list(model.parameters()),
        lr=lr, weight_decay=1e-4,
    )

    out_dir = out_root / f"regime_{state}"
    out_dir.mkdir(parents=True, exist_ok=True)

    best_val   = float("inf")
    train_hist = []
    val_hist   = []

    for epoch in range(1, epochs + 1):
        t_loss = _train_one_epoch(tokenizer, model, train_loader, optimizer, device)
        v_loss = _validate_epoch(tokenizer, model, val_loader, device)
        train_hist.append(t_loss)
        val_hist.append(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            tokenizer.save_pretrained(str(out_dir / "tokenizer"))
            model.save_pretrained(str(out_dir / "model"))
            tag = "  *saved*"
        else:
            tag = ""

        logger.info(
            "  state %d  epoch %d/%d  train=%.5f  val=%.5f%s",
            state, epoch, epochs, t_loss, v_loss, tag,
        )

    return {
        "state": state,
        "type": "regime",
        "n_samples": len(df),
        "best_val_loss": best_val,
        "checkpoint_path": str(out_dir),
        "train_loss_history": train_hist,
        "val_loss_history": val_hist,
    }

print("Fine-tuning function defined.")

In [ ]:
# ── Run fine-tuning for each regime stratum ────────────────────────────────
# This cell will take several minutes per stratum on CPU.
# Set DEVICE="cuda" in your KronosConfig for GPU acceleration.

stratum_results: dict[int, dict] = {}

for state, df_stratum in sorted(strata.items()):
    print(f"\n{'='*60}")
    print(f"  Fine-tuning on regime state {state}  ({len(df_stratum):,} bars)")
    print(f"{'='*60}")
    meta = finetune_stratum(
        state,
        df_stratum,
        cfg_k=cfg_k,
        out_root=OUT_ROOT,
    )
    if meta:
        stratum_results[state] = meta

print("\nDone. Checkpoints saved to:", OUT_ROOT)

## 6. Evaluate: Validation Loss per Regime

In [ ]:
if not stratum_results:
    print("No strata trained — check MIN_STRATUM_SIZE and bar count.")
else:
    n_plots = len(stratum_results)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4), sharey=True)
    if n_plots == 1:
        axes = [axes]

    for ax, (state, meta) in zip(axes, sorted(stratum_results.items())):
        epochs_range = range(1, len(meta["train_loss_history"]) + 1)
        ax.plot(epochs_range, meta["train_loss_history"], label="train", lw=1.5)
        ax.plot(epochs_range, meta["val_loss_history"],   label="val",   lw=1.5, ls="--")
        ax.axhline(meta["best_val_loss"], color="red", lw=1, ls=":", label=f"best={meta['best_val_loss']:.4f}")
        ax.set_title(f"State {state}  (n={meta['n_samples']:,})")
        ax.set_xlabel("Epoch")
        ax.legend(fontsize=7)

    axes[0].set_ylabel("Loss")
    fig.suptitle(f"{SYMBOL} — Kronos val loss per regime stratum", y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary table
    summary = pd.DataFrame([
        {
            "state":           m["state"],
            "n_bars":          m["n_samples"],
            "best_val_loss":   round(m["best_val_loss"], 5),
            "checkpoint":      m["checkpoint_path"],
        }
        for m in stratum_results.values()
    ]).sort_values("state")
    print(summary.to_string(index=False))

## 7. Export Regime Metadata JSON

This file is consumed by `scripts/upload_kronos_checkpoints_hf.py` and can be
used to wire the right checkpoint per regime in `KronosConfig.finetuned_model_path`.

In [ ]:
import datetime

# Attach regime labels from the HMM fit
if "hmm_state_label" in forecast_df.columns:
    label_map = (
        forecast_df.groupby("hmm_state")["hmm_state_label"]
        .agg(lambda x: x.mode().iloc[0])
        .to_dict()
    )
else:
    label_map = {}

metadata = {
    "symbol": SYMBOL,
    "hmm_states": HMM_STATES,
    "generated_at": datetime.datetime.utcnow().isoformat(),
    "model_base": cfg_k.model_name,
    "tokenizer_base": cfg_k.tokenizer_name,
    "hyperparams": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LEARNING_RATE,
        "lookback": LOOKBACK,
        "pred_len": PRED_LEN,
    },
    "checkpoints": [
        {
            "state":            m["state"],
            "label":            label_map.get(m["state"], f"state_{m['state']}"),
            "type":             "regime",
            "n_samples":        m["n_samples"],
            "best_val_loss":    m["best_val_loss"],
            "checkpoint_path":  m["checkpoint_path"],
        }
        for m in sorted(stratum_results.values(), key=lambda x: x["state"])
    ],
}

meta_path = OUT_ROOT / "regime_metadata.json"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
with open(meta_path, "w") as fh:
    json.dump(metadata, fh, indent=2)

print(f"Regime metadata saved to {meta_path}")
print(json.dumps(metadata, indent=2))

## 8. Next Steps

### Use a regime-tuned checkpoint

Set the `finetuned_model_path` in `configs/default.yaml` to point at the
checkpoint for the current live regime:

```yaml
kronos:
  finetuned_model_path: data/models/kronos/SPY/regime_2
```

Or pass it programmatically:

```python
from rlm.kronos.config import KronosConfig
from rlm.pipeline import FullRLMPipeline, FullRLMConfig

cfg = FullRLMConfig(symbol="SPY", use_kronos=True)
pipeline = FullRLMPipeline(cfg)
result = pipeline.run(bars_df)
```

### Upload to Hugging Face

```bash
python scripts/upload_kronos_checkpoints_hf.py \
    --repo-id your-org/kronos-rlm-spy \
    --symbol SPY
```

### Champion-model automation

Wire `calibrate_regime_models.py` output into a weekly cron that:
1. Detects the promoted regime champion
2. Points `KronosConfig.finetuned_model_path` to the matching stratum checkpoint
3. Runs the live pipeline with the updated config